# 第 2 章 经营健康诊断与数据探索

本章商业问题：**GMV 变化到底来自流量、转化、客单价、退款，还是毛利结构？**

本章所有代码默认优先读取真实 ETL 接口 `http://192.168.31.47:38173/api/etl`。如果接口暂时不可用，会自动回退到本地 SQLite 后备数据，保证课堂可以继续进行。

## 0. 先建立商业问题意识

在商业课里，代码不是第一步。第一步是判断：这个问题影响收入、成本、用户体验、库存风险，还是营销效率？只有先说清楚商业目标，后面的数据选择和模型选择才不会跑偏。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if COURSE_ROOT.name in ["notebooks", "student_notebooks", "teacher_notebooks"]:
    COURSE_ROOT = COURSE_ROOT.parent
elif not (COURSE_ROOT / "course_utils").exists():
    COURSE_ROOT = Path("..").resolve()

sys.path.insert(0, str(COURSE_ROOT))

from course_utils.data_loader import (
    API_BASE, load_table, get_metrics, get_quality_report,
    get_table_catalog, get_schema, paid_orders, api_status, query_table
)
from course_utils.business import money, pct, section

try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

print("课程目录:", COURSE_ROOT)
print("ETL API:", API_BASE)
print("API 状态:", api_status())

## 1. 查看 ETL 数据资产

下面先查看 ETL 接口暴露了哪些表。请注意 `dim_` 开头的是维度表，通常描述对象；`fact_` 开头的是事实表，通常记录业务事件。

In [ ]:
catalog = get_table_catalog()
tables = catalog["tables"]
print("可用表数量:", catalog.get("total", len(tables)))
for t in tables[:12]:
    print(t["tableName"], t.get("recordCount"), t.get("type"), t.get("description", ""))

## 2. 从日经营汇总看趋势

经营健康诊断先看趋势，再拆原因。趋势告诉我们什么时候变了，拆因告诉我们该从哪里改。

In [ ]:
import pandas as pd
daily = load_table("daily_business_summary", limit=100000)
print(daily.head())
if "gmv" not in daily.columns:
    orders = paid_orders()
    daily = orders.groupby("order_date").agg(gmv=("paid_amount", "sum"), orders=("order_id", "nunique"), buyers=("user_id", "nunique")).reset_index()
date_source = daily["summary_date"] if "summary_date" in daily.columns else daily.get("date_id", daily.get("order_date"))
daily["analysis_date"] = pd.to_datetime(date_source, errors="coerce")
daily = daily.dropna(subset=["analysis_date"])
daily["month"] = daily["analysis_date"].dt.to_period("M").astype(str)
monthly = daily.groupby("month").agg(gmv=("gmv", "sum"), orders=("orders", "sum")).reset_index()
monthly["aov"] = monthly["gmv"] / monthly["orders"]
monthly.tail()

In [ ]:
import matplotlib.pyplot as plt
monthly.plot(x="month", y=["gmv", "orders"], secondary_y="orders", figsize=(10, 4), title="Monthly GMV and Orders")
plt.xticks(rotation=45)
plt.tight_layout()

## 3. 商业解释

如果 GMV 下降但订单数稳定，优先查客单价和折扣；如果订单数下降，优先查流量和转化；如果 GMV 增长但毛利下降，说明增长质量可能不好。

In [ ]:
assert len(monthly) > 0
print("第 02 章验证通过")

## 学生作业

请补充 150 到 300 字商业解读，至少引用两个数据结果，并给出一个明确运营动作。